# SmartAd Performance Analysis — Classical A/B Testing

---

## 📋 Overview

This notebook implements a **Classical (Frequentist) A/B Hypothesis Test** to determine whether
SmartAd's creative advertisement produced a statistically significant increase in brand awareness
compared to a dummy/control advertisement.

### Business Context

An advertising company, **SmartAd**, runs interactive browser-based ads on behalf of a client.
To justify their premium service — which claims to boost brand awareness beyond simple impressions —
they need a rigorous statistical framework to *quantify the causal lift* from their ads.

They ran an **A/B experiment** from July 3–10, 2020:
- **Control group**: Users shown a standard dummy advertisement
- **Exposed group**: Users shown the client's creative, interactive advertisement

After viewing the ad, users were prompted with a brand awareness survey.
The binary outcome (`yes`=1 / `no`=0) forms the basis of our statistical test.

---

## 🎯 Hypotheses

| | Description |
|---|---|
| **H₀ (Null)** | The creative ad has no effect: p_exposed = p_control |
| **H₁ (Alternative)** | The creative ad increases awareness: p_exposed > p_control |

We test this using a **two-proportion Z-test** with:
- Significance level α = 0.05
- One-sided test (we only care about positive lift)

---

## 🔬 Statistical Framework

### Key Formulas

**Pooled proportion** (under H₀, both groups share the same p):
$$\hat{p} = \frac{X_A + X_B}{N_A + N_B}$$

**Pooled standard error**:
$$SE = \sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{N_A} + \frac{1}{N_B}\right)}$$

**Z-statistic**:
$$z = \frac{\hat{p}_B - \hat{p}_A}{SE}$$

**Minimum detectable effect** (for sample size planning):
$$n = \frac{(z_\alpha + z_\beta)^2 \cdot [p_A(1-p_A) + p_B(1-p_B)]}{(p_B - p_A)^2}$$

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import scipy.stats as scs

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'script'))

from ABTestingFunctions import ABTesting
from HypothesisPlot import HypothesisPlot

ABT = ABTesting()
HP  = HypothesisPlot()

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.dpi': 120})

print('✅ Imports successful')

---
## 📂 1. Load & Prepare Data

In [ ]:
# Load raw data
df_raw = pd.read_csv('../data/AdSmart_AB_testing_data.csv')
print(f'Raw shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# Remove non-respondents (yes=0 AND no=0 means the user ignored the survey)
df = df_raw.query('not (yes == 0 and no == 0)').copy()
df['aware'] = df['yes'].astype(int)

print(f"Non-respondents removed : {len(df_raw) - len(df):,}")
print(f"Respondent rows         : {len(df):,}")
print(f"Response rate           : {len(df)/len(df_raw):.2%}")
df[['experiment', 'aware']].value_counts()

---
## 📊 2. Compute Conversion Rates

The **conversion rate** here is the proportion of survey respondents who answered 'Yes' to the brand awareness question.

In [ ]:
ctrl = df[df['experiment'] == 'control']
exp_ = df[df['experiment'] == 'exposed']

N_C, N_E = len(ctrl), len(exp_)
X_C, X_E = int(ctrl['aware'].sum()), int(exp_['aware'].sum())
p_ctrl = X_C / N_C
p_exp  = X_E / N_E
d_hat  = p_exp - p_ctrl

print(f'Control   → n={N_C:,}, successes={X_C}, conversion rate={p_ctrl:.4f}')
print(f'Exposed   → n={N_E:,}, successes={X_E}, conversion rate={p_exp:.4f}')
print(f'Observed lift (d̂)     = {d_hat:+.4f} ({d_hat/p_ctrl*100:+.1f}% relative)')

---
## ⚖️ 3. Statistical Test

We compute the pooled standard error and then the Z-score for our two-proportion test.

In [ ]:
# Pooled standard error under H₀
SE = ABT.pooled_SE(N_C, N_E, X_C, X_E)

# P-value (one-sided)
p_value = ABT.p_val(N_C, N_E, p_ctrl, p_exp)

# Critical z-value at α=0.05 one-tailed
z_crit = ABT.z_val(sig_level=0.05, two_tailed=False)
z_obs  = d_hat / SE

# 95% Confidence Interval for d_hat
ci_lo, ci_hi = ABT.confidence_interval(d_hat, SE, N_C + N_E, sig_level=0.05)

print('='*50)
print(f'Pooled SE          : {SE:.6f}')
print(f'Observed Z-score   : {z_obs:.4f}')
print(f'Critical Z (α=0.05): {z_crit:.4f}')
print(f'P-value            : {p_value:.4f}')
print(f'95% CI for lift    : [{ci_lo:.4f}, {ci_hi:.4f}]')
print('='*50)
print()

sig = p_value < 0.05
print(f'Decision: {"✅ REJECT H₀ — statistically significant lift detected." if sig else "❌ FAIL TO REJECT H₀ — no significant difference."}')

---
## 📈 4. Hypothesis Distribution Plot

Visualise the null (H₀) and alternative (H₁) sampling distributions, showing
how our observed lift compares to the critical region.

In [ ]:
HP.abplot(
    N_A=N_C, N_B=N_E,
    bcr=p_ctrl, d_hat=d_hat,
    sig_level=0.05,
    show_alpha=True,
    show_power=True,
    show_p_value=True,
    show_legend=True,
)

---
## 🎯 5. Sample Size Planning

How large would our sample need to be to reliably detect various effect sizes?
This is crucial for planning future campaigns.

In [ ]:
# Sample size vs MDE power curve
mde_values = np.arange(0.005, 0.061, 0.005)
n_required_80 = [ABT.min_sample_size(p_ctrl, m, sig_level=0.05, power=0.80) for m in mde_values]
n_required_90 = [ABT.min_sample_size(p_ctrl, m, sig_level=0.05, power=0.90) for m in mde_values]

print(f'Minimum sample size for MDE={d_hat:.4f} at 80% power: {ABT.min_sample_size(p_ctrl, abs(d_hat), 0.05, 0.80):,}')
print(f'Actual sample sizes: Control={N_C:,}, Exposed={N_E:,}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mde_values * 100, n_required_80, 'o-', label='Power = 80%', color='#4361ee')
ax.plot(mde_values * 100, n_required_90, 's--', label='Power = 90%', color='#f72585')
ax.axhline(min(N_C, N_E), color='#2a9d8f', linestyle=':', linewidth=2,
           label=f'Actual min group size ({min(N_C,N_E):,})')
ax.axvline(abs(d_hat) * 100, color='#f4a261', linestyle='--',
           label=f'Observed MDE ({abs(d_hat)*100:.2f}%)')
ax.set_xlabel('Minimum Detectable Effect (%)', fontsize=12)
ax.set_ylabel('Required Sample Size per Group', fontsize=12)
ax.set_title('Sample Size Planning: Power Curve', fontsize=14)
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
## 📊 6. Subgroup Analysis

Examine whether the treatment effect is consistent across different user segments.

In [ ]:
# Map platform_os to readable name
df['os_name'] = df['platform_os'].map({5: 'iOS', 6: 'Android'}).fillna('Other')

# Subgroup conversion rates by OS
subgroup = df.groupby(['os_name', 'experiment'])['aware'].agg(['mean', 'count']).reset_index()
subgroup.columns = ['OS', 'Group', 'Conversion Rate', 'n']
print(subgroup.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
pivot = subgroup.pivot(index='OS', columns='Group', values='Conversion Rate')
pivot.plot(kind='bar', ax=ax, color=['#4361ee', '#f72585'], width=0.6, alpha=0.85)
ax.set_ylabel('Awareness Rate', fontsize=12)
ax.set_title('Conversion Rate by OS and Experiment Group', fontsize=13)
ax.tick_params(axis='x', rotation=0)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.1%}'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

---
## 🏁 7. Conclusions

### Summary of Findings

| Metric | Value |
|---|---|
| Control conversion rate | p_A |
| Exposed conversion rate | p_B |
| Observed lift (d̂) | p_B − p_A |
| P-value | Computed above |
| Statistical significance | Yes/No at α=0.05 |

### Interpretation

The classical A/B test gives us a binary decision at a fixed point in time after collecting the data.
However, this approach has a key limitation: the sample size must be decided *before* running the
experiment, and peeking at results early inflates the Type I error rate.

**Next step**: See `sequential_AB_testing.ipynb` for continuous monitoring via SPRT,
which allows early stopping without inflating error rates.